# Trabalho 2 - Confiabilidade Composta

Este notebook utiliza a biblioteca `pse_confiabilidade` para calcular os indices `LOLP`, `LOLE`, `EPNS` e `EENS` pelos metodos de enumeracao de estados e simulacao Monte Carlo nao sequencial.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from pse_confiabilidade import (
    MonteCarloConfig,
    build_default_case,
    enumerate_indices,
    simulate_indices,
)


In [ ]:
case = build_default_case()

print("Sistema estudado")
print(f"Gerador: {case.generator.installed_capacity_mw:.0f} MW")
print(f"LTs: {len(case.transmission_lines)} x {case.transmission_lines[0].installed_capacity_mw:.0f} MW")
print(f"Total de blocos: {len(case.time_blocks)}")
print(f"Horas totais: {case.total_hours:.0f}")

print("\nDisponibilidades estacionarias")
print(f"Gerador: A = {case.generator.availability:.9f}, U = {case.generator.unavailability:.9f}")
for line in case.transmission_lines:
    print(f"{line.name}: A = {line.availability:.9f}, U = {line.unavailability:.9f}")


In [ ]:
print("Blocos anuais")
print(f"{'Bloco':<6} {'Horas':>8} {'Energia disp. (MW)':>20} {'Carga (MW)':>12}")
for index, block in enumerate(case.time_blocks, start=1):
    print(
        f"{index:<6} {block.duration_hours:>8.0f} {block.energy_availability_mw:>20.0f} {block.load_mw:>12.0f}"
    )


In [ ]:
def print_indices(title, indices):
    print(title)
    print(f"  LOLP : {indices.lolp:.9f}")
    print(f"  LOLE : {indices.lole_hours_per_year:.6f} h/ano")
    print(f"  EPNS : {indices.epns_mw:.6f} MW")
    print(f"  EENS : {indices.eens_mwh_per_year:.6f} MWh/ano")


def print_comparison(reference, estimate):
    print(f"{'Indice':<6} {'Erro abs.':>14} {'Erro rel.':>14}")
    for label, ref_value, est_value in (
        ("LOLP", reference.lolp, estimate.lolp),
        ("LOLE", reference.lole_hours_per_year, estimate.lole_hours_per_year),
        ("EPNS", reference.epns_mw, estimate.epns_mw),
        ("EENS", reference.eens_mwh_per_year, estimate.eens_mwh_per_year),
    ):
        absolute_error = abs(est_value - ref_value)
        relative_error = absolute_error / ref_value if ref_value else 0.0
        print(f"{label:<6} {absolute_error:>14.6f} {relative_error:>13.6%}")


In [ ]:
enumeration_indices = enumerate_indices(case)
print_indices("Resultados por enumeracao de estados", enumeration_indices)


In [ ]:
smc_config = MonteCarloConfig(
    seed=42,
    target_cv_epns=0.05,
    min_samples=20_000,
    batch_size=5_000,
    max_samples=200_000,
)

smc_result = simulate_indices(case, smc_config)
print_indices("Resultados por SMC nao sequencial", smc_result.indices)
print(f"\nAmostras: {smc_result.samples}")
print(f"CV do EPNS: {smc_result.cv_epns:.6f}")
print(f"Seed: {smc_result.seed}")


In [ ]:
print("Comparacao entre SMC e enumeracao")
print_comparison(enumeration_indices, smc_result.indices)


## Observacao

Neste caso, a enumeracao fornece a referencia exata do modelo adotado. A SMC nao sequencial aproxima esses valores e permite analisar a convergencia pela quantidade de amostras e pelo coeficiente de variacao do `EPNS`.
